# Preprocessing & Data Diagnostics: Messy POS Data
# 预处理与数据诊断：杂乱 POS 数据

Industrial time series are rarely clean. This tutorial creates duplicate timestamps, missing dates, outliers, and stockout artifacts, then uses PipelineTS preprocessing and diagnostic APIs.

工业时间序列很少天然干净。本教程构造重复时间戳、缺失日期、异常值和缺货影响，然后使用 PipelineTS 预处理与诊断 API。

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

def make_retail_demand(n_days=240, n_stores=1, start="2023-01-01"):
    rows = []
    for i in range(n_stores):
        rng = np.random.default_rng(100 + i)
        dates = pd.date_range(start, periods=n_days, freq="D")
        dow = dates.dayofweek.to_numpy()
        month = dates.month.to_numpy()
        holiday = ((dow >= 5) | rng.binomial(1, 0.04, n_days).astype(bool)).astype(int)
        promotion = rng.binomial(1, 0.14 + 0.06 * (dow >= 4), n_days).astype(int)
        price_index = 1.0 + 0.04 * np.sin(np.linspace(0, 5 * np.pi, n_days)) + rng.normal(0, 0.015, n_days)
        temperature = 18 + 10 * np.sin(np.linspace(-0.8, 2.8 * np.pi, n_days)) + rng.normal(0, 1.8, n_days)
        stockout = rng.binomial(1, 0.025, n_days)
        baseline = 120 + 18 * i
        weekly = np.where(dow < 5, 8, 28)
        seasonal = 16 * np.sin(2 * np.pi * np.arange(n_days) / 365.25 + i / 3)
        trend = 0.08 * np.arange(n_days)
        demand = (
            baseline + weekly + seasonal + trend
            + 34 * promotion + 22 * holiday
            + 0.9 * np.maximum(temperature - 20, 0)
            - 75 * (price_index - 1.0)
            - 45 * stockout
            + rng.normal(0, 7, n_days)
        )
        rows.append(pd.DataFrame({
            "date": dates,
            "store_id": f"store_{i + 1:02d}",
            "sales": np.maximum(demand, 1),
            "promotion": promotion,
            "holiday": holiday,
            "price_index": price_index,
            "temperature": temperature,
            "stockout": stockout,
            "month": month,
        }))
    return pd.concat(rows, ignore_index=True)

In [ ]:
raw = make_retail_demand(n_days=120, n_stores=1).drop(columns=["store_id"])
messy = raw.copy()
messy = pd.concat([messy, messy.iloc[[20, 21]]], ignore_index=True)
messy.loc[10:12, "sales"] = np.nan
messy.loc[45, "sales"] *= 4
messy = messy.drop(index=[30, 31, 32]).sample(frac=1, random_state=42).reset_index(drop=True)
messy.head()

In [ ]:
from PipelineTS.preprocessing import TimeSeriesDataQualityReport

quality = TimeSeriesDataQualityReport(time_col="date", target_col="sales")
report = quality.fit(messy)
report["overview"], report["issues"][:5]

In [ ]:
from PipelineTS.preprocessing import sort_and_deduplicate, resample_time_series, clip_or_winsorize, smooth_series

clean = sort_and_deduplicate(messy, time_col="date", agg="mean")
clean = resample_time_series(clean, time_col="date", target_col="sales", freq="D", fill_method="linear")
clean = clip_or_winsorize(clean, target_col="sales", lower_q=0.01, upper_q=0.99)
clean = smooth_series(clean, target_col="sales", method="rolling_mean", window=3)
clean.head()

In [ ]:
from PipelineTS.preprocessing import TimeSeriesPreprocessor

prep = TimeSeriesPreprocessor()
transformed = prep.transform_target(clean, target_col="sales", method="log1p")
differenced = prep.difference_series(clean, target_col="sales", order=1)

display(transformed.head())
display(differenced.head())

In [ ]:
from PipelineTS.preprocessing import (
    time_index_report,
    series_profile,
    forecastability_report,
    baseline_forecast_report,
    leakage_risk_report,
    modeling_readiness_report,
)

print(time_index_report(clean, time_col="date"))
print(series_profile(clean, target_col="sales"))
print(forecastability_report(clean, target_col="sales", horizon=14))
print(baseline_forecast_report(clean, time_col="date", target_col="sales", horizon=14))
print(leakage_risk_report(clean, time_col="date", target_col="sales", known_covariates=["promotion", "holiday"]))
print(modeling_readiness_report(clean, time_col="date", target_col="sales"))

In [ ]:
from PipelineTS.plot import plot_series, plot_decomposition, plot_acf_pacf

plot_series(clean, time_col="date", target_col="sales", title="Cleaned POS demand", lang="zh")
plot_decomposition(clean, time_col="date", target_col="sales", lang="zh")
plot_acf_pacf(clean["sales"].dropna().values, max_lags=30, lang="zh")